<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week7_day3_daily_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Étape 1 — Environnement


In [1]:
# On installe tout sauf torch/torchvision qui sont déjà bien installés dans Colab
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters
!pip install -q transformers sentence-transformers datasets faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.0 MB/s eta 0:00:00


Étape 2 — Chargement du dataset


In [2]:
from langchain_community.document_loaders.hugging_face_dataset import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()
print(f"✅ {len(data)} documents chargés")
print(data[:2])

/tmp/ipykernel_483/3117967400.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.hugging_face_dataset import HuggingFaceDatasetLoader


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

✅ 15011 documents chargés
[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


Étape 3 — Splitting


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)
print(docs[0])

page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


Étape 4 — Embeddings


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test rapide
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(f"✅ Succès ! Dimension de l'embedding : {len(query_result)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Succès ! Dimension de l'embedding : 384


Étape 5 — Vector store FAISS


In [5]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)
print(f"✅ Index FAISS créé avec {db.index.ntotal} vecteurs")

✅ Index FAISS créé avec 18502 vecteurs


Étape 6 — LLM génératif

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# On utilise un modèle CausalLM (text-generation) au lieu de Seq2Seq
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("⏳ Téléchargement du tokenizer et du modèle (cela peut prendre 1-2 minutes)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# On utilise la tâche "text-generation" qui est reconnue par votre version de transformers
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Création du wrapper LangChain
llm = HuggingFacePipeline(pipeline=generator)
print("✅ Modèle LLM chargé et prêt !")

⏳ Téléchargement du tokenizer et du modèle (cela peut prendre 1-2 minutes)...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Modèle LLM chargé et prêt !


Étape 7 — RetrievalQA

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import textwrap

# 1. On crée un prompt clair pour le modèle génératif
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context below to answer the question.
If you don't know the answer based on the context, just say "I don't know."

Context:
{context}

Question: {question}

Answer:""")

# 2. Fonction pour formater les documents récupérés en un seul texte
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 3. On construit la chaîne RAG avec LCEL
retriever = db.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    # On récupère le contexte via le retriever, et on passe la question telle quelle
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test du système
question = "What is cheesemaking?"
print("❓ Question :", question)
print("\n⏳ Génération de la réponse en cours...\n")

# On invoque la chaîne
answer = rag_chain.invoke(question)

print("🤖 Réponse :\n", textwrap.fill(answer, 100))

❓ Question : What is cheesemaking?

⏳ Génération de la réponse en cours...



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🤖 Réponse :
 Human:  You are a helpful assistant. Use ONLY the context below to answer the question. If you don't
know the answer based on the context, just say "I don't know."  Context: "The goal of cheese making
is to control the spoiling of milk into cheese. The milk is traditionally from a cow, goat, sheep or
buffalo, although, in theory, cheese could be made from the milk of any mammal. Cow's milk is most
commonly used worldwide. The cheesemaker's goal is a consistent product with specific
characteristics (appearance, aroma, taste, texture). The process used to make a Camembert will be
similar to, but not quite the same as, that used to make Cheddar.\n\nSome cheeses may be
deliberately left to ferment from naturally airborne spores and bacteria; this approach generally
leads to a less consistent product but one that is valuable in a niche market.\n\nCulturing\nCheese
is made by bringing milk (possibly pasteurised) in the cheese vat to a temperature required to
promote the growth o

 Architecture RAG Moderne (LCEL)

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context below to answer the question.
If you don't know, say "I don't know based on the provided context."

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What is cheesemaking?")
print(answer)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: 
You are a helpful assistant. Use ONLY the context below to answer the question.
If you don't know, say "I don't know based on the provided context."

Context:
"The goal of cheese making is to control the spoiling of milk into cheese. The milk is traditionally from a cow, goat, sheep or buffalo, although, in theory, cheese could be made from the milk of any mammal. Cow's milk is most commonly used worldwide. The cheesemaker's goal is a consistent product with specific characteristics (appearance, aroma, taste, texture). The process used to make a Camembert will be similar to, but not quite the same as, that used to make Cheddar.\n\nSome cheeses may be deliberately left to ferment from naturally airborne spores and bacteria; this approach generally leads to a less consistent product but one that is valuable in a niche market.\n\nCulturing\nCheese is made by bringing milk (possibly pasteurised) in the cheese vat to a temperature required to promote the growth of the bacteria that 